# Introduction to Data Analysis and Modeling with Python

In this notebook, we will learn the first steps that data scientists and machine learning engineers use when working with numerical data.

We will practice:

- Loading a dataset
- Inspecting the data
- Cleaning missing values
- Using NumPy and Pandas
- Visualizing data
- Finding correlations
- Filtering data
- Fitting simple models
- Evaluating model performance
- Making predictions

By the end, you should understand the basic workflow used before building machine learning models.

We use:

- `pandas` to load and organize data
- `numpy` for numerical calculations
- `matplotlib` for plotting

In [ ]:
# Although there are some exceptions, it is generally a good idea to keep all of your
# imports in one place so that you can easily manage them. Doing so also makes it easy
# to copy all of them at once and paste them into a new notebook you are starting.

import numpy as np # NumPy is a library for working with arrays and performing mathematical operations on them.
import pandas as pd # Pandas is a library for working with data in tabular form, such as data frames.
import matplotlib.pyplot as plt # Matplotlib is a library for creating visualizations, such as plots and charts.

## 1. Loading the Dataset

The first step in any data analysis project is loading the data.

Our dataset contains yearly GDP values for different countries.

If you open the CSV file, you will notice that the first 4 rows contain extra information that is not part of the actual data table. Because of this, we use `skiprows=4`.

In [ ]:
gdp = pd.read_csv("GDP_Data.csv", delimiter=",", skiprows=4) # Make sure the csv file and notebook are in the same folder, or provide the full path to the csv file.

gdp.head() # The head() method is used to display the first few rows of a DataFrame. By default, it shows the first 5 rows, but you can specify a different number by passing an argument to the method (e.g., gdp.head(10) to show the first 10 rows).

`head()` shows the first few rows of the dataset. This helps us check that the file loaded correctly.

## 2. First Look at the Data

Before doing any analysis, we should understand what kind of data we have.

Important questions:

- How many rows and columns are there?
- What are the column names?
- Which columns are numerical?
- Are there missing values?
- What are the minimum, maximum, mean, and standard deviation?

In [ ]:
gdp.shape # Shows the number of rows and columns in the DataFrame

In [ ]:
gdp.columns # Shows the column names in the DataFrame

In [ ]:
gdp.info() # The info() method provides a concise summary of the DataFrame, including the number of non-null entries, data types of each column, and memory usage. 

## 3. Finding Missing Values

Real-world datasets often contain missing values.

In Python, missing numerical values usually appear as `NaN`, which means “Not a Number.”

Before cleaning the data, we should check where those missing values are.

In [ ]:
gdp.isna().sum() # The isna() method returns a DataFrame of the same shape as the original, with True for entries that are NaN (missing) and False for entries that are not

The dataset also contains a fully empty column at the end called something like `Unnamed: 67`.

If we use `dropna()` immediately, Python will remove every row because every row has a missing value in that empty column.

So we first remove fully empty and unnecessary columns, then remove rows that still contain missing values.

In [ ]:
# Drop columns that are completely empty
gdp_clean = gdp.dropna(axis="columns", how="all")

# Remove unnecessary columns
gdp_clean = gdp_clean.drop(columns=["Country Code", "Indicator Name", "Indicator Code"])

# Drop rows that still contain missing values
gdp_clean = gdp_clean.dropna()

# Check result
gdp_clean.head()

In [ ]:
gdp_clean.shape

In [ ]:
gdp_clean.isna().sum()

### Transposing the Dataset

Now that our dataset only contains:
- Country Name
- GDP values by year

We can transpose the data to better analyze trends over time.

After transposing:
- Rows → Years  
- Columns → Countries

In [ ]:
# Transpose the dataset
gdp_transposed = gdp_clean.set_index("Country Name").T # Transpose the dataset and set "Country Name" as the index

# Convert index (years) to integers
gdp_transposed.index = gdp_transposed.index.astype(int) # Convert the index to integers so that it can be used for plotting and analysis

# Check result
gdp_transposed.head()

## 4. Basic Statistics

Now that the data has been cleaned, we can calculate basic statistics.

In [ ]:
gdp_transposed.describe() # The describe() method generates descriptive statistics of the DataFrame, such as count, mean, standard deviation, minimum, and maximum values for each column. 

Or use .describe() for a specific country.

In [ ]:
country = "Australia"

gdp_transposed[country].describe()

Another way to calculate basic statistics is using numpy (np).

In [ ]:
gdp_Australia = gdp_transposed[country]

print("Mean GDP:", np.mean(gdp_Australia)) 
print("Median GDP:", np.median(gdp_Australia))
print("Standard deviation:", np.std(gdp_Australia))

The mean gives the average value.

The median gives the middle value.

The standard deviation tells us how spread out the values are.

## 5. Visualizing One Variable

Here, we will look at how GDP values for a single country are distributed over time.

In [ ]:
country = "Australia"  # or any country you want

gdp_country = gdp_transposed[country] # Extract the GDP data for the specified country from the transposed DataFrame

plt.figure(figsize=(8,5)) # Create a new figure with a specified size (width=8 inches, height=5 inches) for better visualization of the plot
plt.plot(gdp_country) # Plot the GDP data for the specified country. x-axis is not explicitly specified, so it will default to the index of the DataFrame, which represents the years, and the y-axis will be the GDP values for that country. 
plt.xlabel("Year Index") # Set the label for the x-axis to "Year Index". This indicates that the x-axis represents the years corresponding to the GDP data.
plt.ylabel("GDP [current US$]") # Set the label for the y-axis to "GDP [current US$]". This indicates that the y-axis represents the GDP values in current US dollars.
plt.title(f"GDP Over Time for {country}") # Set the title for the plot to "GDP Over Time for [Country]".

plt.grid(True) # Add a grid to the plot for better readability.
plt.show() # Display the plot. 

## 6. Filtering Data with Masks

Sometimes we only want to look at specific parts of our data.

In Python, we can use **masks** (boolean conditions) to filter data.

For example, we might want to:
- Look at years where GDP is above a certain value
- Focus on periods of rapid growth

Japan, for exmaple, shows relatively consistent growth up to around 1995, as shown below.

In [ ]:
country = "Japan"
gdp_japan = gdp_transposed[country]

plt.figure(figsize=(8,5))
plt.plot(gdp_japan, label="GDP")

plt.xlabel("Year Index")
plt.ylabel("GDP [current US$]")
plt.title(f"GDP Over Time for {country}")

plt.legend()
plt.grid(True)
plt.show()

Using a mask, we can isolate specific time periods in the data.

This allows us to focus on particular trends without removing the rest of the dataset.

In [ ]:
# Create mask for years before 1995
mask = gdp_japan.index < 1995 # Create a boolean mask that is True for years before 1995 and False for years 1995 and later.

gdp_before_1995 = gdp_japan[mask] # Use the mask to filter the GDP data for years before 1995.

In [ ]:
plt.figure(figsize=(8,5))

#plt.plot(gdp_japan, label="All data")
plt.plot(gdp_before_1995, label="Before 1995", linewidth=3)

plt.xlabel("Year")
plt.ylabel("GDP [current US$]")
plt.title(f"GDP Growth for {country} (Highlighting Pre-1995)")

plt.legend()
plt.grid(True)
plt.show()

## 7. Comparing Multiple Countries

Now we will compare GDP over time for several countries.

In [ ]:
countries = ["United States", "China", "Japan", "Canada"] # Create a list of countries that we want to plot.

years = gdp_transposed.index.astype(int) # Convert the index to integers so that it can be used for plotting and analysis. This will allow us to plot the GDP data against the actual years on the x-axis.

plt.figure(figsize=(10, 6))

for country in countries: # Loop through each country in the list of countries to plot their GDP data on the same graph for comparison.
    if country not in gdp_transposed.columns: # Check if the country is present in the columns of the transposed DataFrame. If not, print a warning message and skip to the next country in the loop.
        print(f"Warning: {country} not found in dataset")
        continue

    gdp_values = gdp_transposed[country].values.astype(float) # Extract the GDP values for the current country and convert them to floats for plotting. 
    plt.plot(years, gdp_values, label=country) # Plot the GDP values for the current country against the years. 

plt.xlabel("Year")
plt.ylabel("GDP [current US$]")
plt.title("GDP Over Time for Selected Countries")

plt.legend()
plt.grid(True)
plt.show()

When values are very different in size, a log scale can make comparison easier.

In [ ]:
plt.figure(figsize=(10, 6))

for country in countries:
    if country not in gdp_transposed.columns:
        continue

    gdp_values = gdp_transposed[country].values.astype(float)
    plt.plot(years, gdp_values, label=country)

plt.yscale("log") # Set the y-axis to a logarith scale.

plt.xlabel("Year")
plt.ylabel("GDP [current US$] — log scale")
plt.title("GDP Over Time for Selected Countries (Log Scale)")

plt.legend()
plt.grid(True)
plt.show()

## 8. Correlation

Correlation measures how strongly two numerical variables are related.

A correlation close to 1 means the variables tend to increase together.

A correlation close to -1 means one variable tends to decrease as the other increases.

A correlation close to 0 means there is not a strong linear relationship.

In [ ]:
selected_countries = [
    "United States",
    "China",
    "Japan",
    "Canada",
    "Australia",
    "Austria"
]

# Select only these countries from the transposed dataset
country_data = gdp_transposed[selected_countries]

# Compute correlation matrix
corr_matrix = country_data.corr()

corr_matrix

In [ ]:
plt.figure(figsize=(7, 6))

plt.imshow(corr_matrix)

plt.colorbar()

plt.xticks(
    range(len(selected_countries)),
    selected_countries,
    rotation=45 # Rotate the x-axis labels by 45 degrees for better readability.
)

plt.yticks(
    range(len(selected_countries)),
    selected_countries
)

plt.title("Correlation Matrix Between Countries")

plt.show()

### Interpreting the Correlation Matrix

Countries with values closer to 1 have more similar GDP trends over time.

This does not mean the countries are identical, but it suggests their economies may grow in similar ways.

### Correlation Between All Countries

We can compute correlations for every country in the dataset.

However, with more than 100 countries, the correlation matrix becomes very difficult to interpret visually.

Even though patterns exist, it is hard to identify specific relationships directly from the heatmap.

In [ ]:
all_corr = gdp_transposed.corr()

plt.figure(figsize=(12,10))
plt.imshow(all_corr)

plt.colorbar()
plt.title("Correlation Matrix for All Countries")

plt.show()

### Limitation of Large Correlation Matrices

As datasets grow larger, visualization alone becomes less useful.

Instead of trying to inspect every country manually, we often use programming tools to:
- sort correlations
- filter results
- identify the strongest relationships automatically

### Finding the Most Correlated Countries

Instead of looking at every country at once, we can focus on one country and compute its correlation with all others.

Here, we will find the countries whose GDP trends are most strongly correlated with Japan.

In [ ]:
country = "Japan"

correlations = gdp_transposed.corr()[country] # Compute the correlation of the GDP data for the specified country with the GDP data of all other countries in the transposed DataFrame.

correlations = correlations.drop(country) # Drop the correlation of the country with itself.

correlations.sort_values(ascending=False) # Sort the correlations in descending order to see which countries have the highest correlation with the specified country.

## 9. Scatter Plots

We can use a scatter plot to compare the GDP of two countries over time.

Each point represents a year:
- x-axis → GDP of Country A
- y-axis → GDP of Country B

If the points follow a clear pattern, it suggests a relationship between their economies.

In [ ]:
country1 = "United States"
country2 = "China"

gdp_1 = gdp_transposed[country1]
gdp_2 = gdp_transposed[country2]

plt.figure(figsize=(8, 5))
plt.scatter(gdp_1, gdp_2)

plt.xlabel(f"{country1} GDP [current US$]")
plt.ylabel(f"{country2} GDP [current US$]")
plt.title(f"{country1} vs {country2} GDP Over Time")

plt.grid(True)
plt.show()

### Interpretation

Each point is a year.

If the points move upward together, it means both countries’ GDPs are increasing over time.

A clear upward trend suggests a strong relationship between the two economies.

## 10. Introduction to Modeling

A model is a simplified way to describe a relationship in data.

For example, we might ask:

Can we use the year to predict GDP?

This is the basic idea behind machine learning:

Use data to learn a pattern, then use that pattern to make predictions.

In [ ]:
from sklearn.linear_model import LinearRegression # Import the LinearRegression class from the linear_model module of scikit-learn, which is a machine learning library in Python. 
from sklearn.metrics import mean_squared_error, r2_score # Import the mean_squared_error and r2_score functions from the metrics module of scikit-learn, which are used to evaluate the performance of regression models.

## 11. Linear Regression

A linear regression model assumes the data follows a straight-line pattern.

In [ ]:
country_name = "United States"

# X = years
years = gdp_transposed.index.to_numpy() # Convert index to numpy array.
X = years.reshape(-1, 1) # Reshape to make it a 2D array with one column, which is required by scikit-learn models.

# y = GDP values for selected country
y = gdp_transposed[country_name].values.astype(float) # Convert to float in case there are any non-numeric values.

linear_model = LinearRegression() # Create an instance of the LinearRegression model from scikit-learn.
linear_model.fit(X, y) # Fit the linear regression model to the data. This will find the best-fitting line that describes the relationship between the years (X) and the GDP values (y) for the specified country.

y_pred_linear = linear_model.predict(X) # Use the fitted linear model to predict the GDP values based on the years (X). This will give us the predicted GDP values according to the linear model, which we can then compare to the actual GDP values (y) to evaluate the model's performance.

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(X, y, label="Actual data")
plt.plot(X, y_pred_linear, color="red", label="Linear model")  

plt.xlabel("Year")
plt.ylabel("GDP [current US$]")
plt.title(f"Linear Model for {country_name} GDP")

plt.legend()
plt.grid(True)
plt.show()

## 12. Evaluating the Linear Model

We need to measure how well the model fits the data.

Two common metrics are:

- RMSE: average prediction error
- R²: how much variation in the data is explained by the model

In [ ]:
rmse_linear = np.sqrt(mean_squared_error(y, y_pred_linear)) # Calculate the Root Mean Squared Error (RMSE) for the linear model by comparing the actual GDP values (y) with the predicted GDP values (y_pred_linear).
r2_linear = r2_score(y, y_pred_linear) # Calculate the R² score for the linear model, which is a measure of how well the model explains the variance in the data. 

print("Linear RMSE:", rmse_linear)
print("Linear R²:", r2_linear)

## 13. Polynomial Regression

A straight line may not describe the data well.

A polynomial model can curve, so it may fit some patterns better.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures # Import the PolynomialFeatures class from the preprocessing module of scikit-learn, which is used to generate polynomial features from the original features. 
from sklearn.pipeline import make_pipeline # Import the make_pipeline function from the pipeline module of scikit-learn, which is used to create a pipeline that combines multiple steps (e.g., feature transformation and model fitting) into a single object that can be treated as a single estimator.

poly_model = make_pipeline( # Create a pipeline that first generates polynomial features of degree 2 from the original features (years) and then fits a linear regression model to those polynomial features.
    PolynomialFeatures(degree=2),
    LinearRegression()
)

poly_model.fit(X, y)

y_pred_poly = poly_model.predict(X)

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(years, y, label="Actual data")
plt.plot(years, y_pred_linear, color="red", label="Linear model")
plt.plot(years, y_pred_poly, color="orange", label="Polynomial model")
plt.xlabel("Year")
plt.ylabel("GDP [current US$]")
plt.title(f"Comparing Models for {country_name} GDP")
plt.legend()
plt.grid(True)
plt.show()

## 14. Comparing Models

Now we compare the linear model and the polynomial model using RMSE and R².

In [ ]:
rmse_poly = np.sqrt(mean_squared_error(y, y_pred_poly))
r2_poly = r2_score(y, y_pred_poly)

print("Linear RMSE:", rmse_linear)
print("Polynomial RMSE:", rmse_poly)

print("Linear R²:", r2_linear)
print("Polynomial R²:", r2_poly)

A lower RMSE means the model has smaller prediction errors.

A higher R² usually means the model explains more of the variation in the data.

However, a model that fits old data very well is not always better at predicting new data.

## 15. Making Predictions

Now we can use our models to predict GDP in a future year.

In [ ]:
future_year = np.array([[2030]])

linear_prediction = linear_model.predict(future_year)
poly_prediction = poly_model.predict(future_year)

print("Linear model prediction for 2030:", linear_prediction[0])
print("Polynomial model prediction for 2030:", poly_prediction[0])

In [ ]:
plt.figure(figsize=(8,5))

plt.scatter(years, y, label="Actual data")
plt.plot(years, y_pred_linear, color="red", linestyle="--", label="Linear model")
plt.plot(years, y_pred_poly, color="orange", linestyle="--", label="Polynomial model")

# Add prediction point
plt.scatter(2030, linear_prediction, color="red", s=100, label="LinearPrediction (2030)")
plt.scatter(2030, poly_prediction, color="orange", s=100, label="Polynomial Prediction (2030)")

plt.xlabel("Year")
plt.ylabel("GDP [current US$]")
plt.title(f"GDP Prediction for {country_name}")

plt.legend()
plt.grid(True)
plt.show()

### Reflection

- Does the predicted value for 2030 follow the trend shown in the plot?
- Which model (linear or polynomial) seems more realistic? Why?
- What might happen if we try to predict too far into the future?

### Important Idea

When we predict the future using past data, we assume that:
- The same patterns will continue
- No major changes occur 

This is why predictions should always be interpreted carefully.

## 16. Training and Testing

A model can look good on the data it was trained on but perform poorly on new data.

To test this, we can train the model on earlier years and test it on later years.

In [ ]:
train_mask = years <= 2010
test_mask = years > 2010

X_train = years[train_mask].reshape(-1, 1)
y_train = y[train_mask]

X_test = years[test_mask].reshape(-1, 1)
y_test = y[test_mask]

In [ ]:
test_model = LinearRegression()
test_model.fit(X_train, y_train)

test_predictions = test_model.predict(X_test)

In [ ]:
plt.figure(figsize=(8, 5))

# Training data
plt.scatter(X_train, y_train, label="Training data", alpha=0.6)

# Actual test data
plt.scatter(X_test, y_test, color="orange", label="Actual (test)")

# Predicted test data (as dots)
plt.scatter(X_test, test_predictions, color="red", label="Predictions")

plt.xlabel("Year")
plt.ylabel("GDP [current US$]")
plt.title(f"Linear Model Performance on Test Data: {country_name}")

plt.legend()
plt.grid(True)
plt.show()

In [ ]:
test_rmse = np.sqrt(mean_squared_error(y_test, test_predictions))
test_r2 = r2_score(y_test, test_predictions)

print("Test RMSE:", test_rmse)
print("Test R²:", test_r2)

In [ ]:
plt.figure(figsize=(6, 6))

plt.scatter(y_test, test_predictions)

# Perfect prediction line
min_val = min(y_test.min(), test_predictions.min())
max_val = max(y_test.max(), test_predictions.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--') # Add a dashed red line for perfect predictions

plt.xlabel("Actual GDP")
plt.ylabel("Predicted GDP")
plt.title("Predicted vs Actual GDP")

plt.grid(True)
plt.show()

This is closer to how machine learning models are evaluated.

We do not only care about how well a model fits old data.

We care about how well it predicts new data.

## 17. Your Turn: Test a Polynomial Model

In the previous section, we trained a **linear regression model** on earlier years and tested it on later years.

Now your task is to repeat the same idea, but using a **polynomial model**.

You can copy and modify code from the previous sections.

### Your goal:

1. Choose a country
2. Split the data into training years and testing years
3. Create a polynomial model
4. Fit the model using the training data
5. Predict the testing years
6. Plot:
   - training data
   - actual testing data
   - predicted testing data
7. Calculate RMSE and R²

### Questions:

1. Which country did you choose?
2. Does the polynomial model predict the test data better than the linear model?
3. Is the R² value better or worse?
4. Do you trust the polynomial model? Why or why not?

In [ ]:
# Choose a country
chosen_country = "United States"

# Get years and GDP values
years = gdp_transposed.index.to_numpy()
y = gdp_transposed[chosen_country].values.astype(float)

# Create training and testing masks
train_mask = years <= 2010
test_mask = years > 2010

# Create X and y training/testing data
X_train = years[train_mask].reshape(-1, 1)
y_train = y[train_mask]

X_test = years[test_mask].reshape(-1, 1)
y_test = y[test_mask]

# Create a polynomial model
# Hint: use make_pipeline(), PolynomialFeatures(), and LinearRegression()
poly_test_model = ...

# Fit the model
...

# Predict the testing data
poly_test_predictions = ...

# Plot training, actual testing, and predicted testing data
plt.figure(figsize=(8, 5))

# Uncomment and fill in the code below to plot the training data, actual test data, and polynomial predictions on the same graph for comparison.
#plt.scatter(..., ..., label="Training data", alpha=0.6)
#plt.scatter(..., ..., color="orange", label="Actual (test)")
#plt.scatter(..., ..., color="red", label="Polynomial predictions")

plt.xlabel("Year")
plt.ylabel("GDP [current US$]")
plt.title(f"Polynomial Model Performance on Test Data: {chosen_country}")

plt.legend()
plt.grid(True)
plt.show()

# Evaluate the model
poly_test_rmse = ...
poly_test_r2 = ...

print("Polynomial Test RMSE:", poly_test_rmse)
print("Polynomial Test R²:", poly_test_r2)

In [ ]:
# Hint for creating the polynomial model:
# poly_test_model = make_pipeline(
#     PolynomialFeatures(degree=2),
#     LinearRegression()
# )

## 18. Optional: Introduction to SVD

Singular Value Decomposition, or SVD, is a method that breaks a dataset into important patterns.

It is often used for:
- Dimensionality reduction
- Compression
- Finding hidden patterns in data
- Preparing data for machine learning

Here, we will apply SVD to the GDP dataset to see if a few patterns can represent most of the data.

In [ ]:
# Select only the numerical GDP values
# Rows = years, columns = countries
gdp_matrix = gdp_transposed.values.astype(float)

# Scale the data so very large countries do not dominate too much
gdp_matrix_scaled = (gdp_matrix - gdp_matrix.mean(axis=0)) / gdp_matrix.std(axis=0)

# Replace possible NaN values caused by zero standard deviation
gdp_matrix_scaled = np.nan_to_num(gdp_matrix_scaled)

SVD separates the data matrix into three parts:

\[
X = U \Sigma V^T
\]

The most important values are the singular values. Larger singular values represent stronger patterns in the data.

In [ ]:
U, S, Vt = np.linalg.svd(gdp_matrix_scaled, full_matrices=False)

S

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(S, marker="o")

plt.xlabel("Component Number")
plt.ylabel("Singular Value")
plt.title("Singular Values of GDP Dataset")

plt.grid(True)
plt.show()

### Interpreting the Plot

If the first few singular values are much larger than the rest, then a small number of patterns explain most of the dataset.

This is useful because it means we may not need every original variable to understand the main structure of the data.

In [ ]:
# Keep only the first k patterns
k = 5

U_k = U[:, :k]
S_k = np.diag(S[:k])
Vt_k = Vt[:k, :]

gdp_approx = U_k @ S_k @ Vt_k

The reconstructed dataset `gdp_approx` is an approximation of the original data using only the first few SVD components.

In [ ]:
country = "Japan"

country_index = list(gdp_transposed.columns).index(country)

original = gdp_matrix_scaled[:, country_index]
approx = gdp_approx[:, country_index]

plt.figure(figsize=(8, 5))

plt.plot(gdp_transposed.index, original, label="Original scaled GDP")
plt.plot(gdp_transposed.index, approx, label=f"SVD approximation (k={k})")

plt.xlabel("Year")
plt.ylabel("Scaled GDP")
plt.title(f"SVD Approximation for {country}")

plt.legend()
plt.grid(True)
plt.show()

### Key Idea

SVD helps us summarize a large dataset using fewer patterns.

This connects to machine learning because many ML methods work better when data is simplified, compressed, or transformed into more meaningful features.

## Final Reflection

In this notebook, we practiced the basic workflow used in data analysis and machine learning:

1. Load data
2. Inspect data
3. Clean missing values
4. Visualize patterns
5. Look for relationships
6. Filter useful subsets
7. Fit a model
8. Evaluate the model
9. Make predictions
10. Question whether the predictions make sense
11. Introduction to SVD

This workflow is the foundation for more advanced machine learning.